# Correct assignments - wave 2

In [1]:
# Set up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
import os
from lab_assignment import assign_enumerators

In [2]:
# Load datasets
existing_assignments = pd.read_csv(config.WAVE2_ENUMERATORS / "assignedlabs.csv")
labs_to_remove_from_sample = pd.read_excel(config.WAVE2_LABS_LIST / "labs_to_remove_from_sample.xlsx")
corrections = pd.read_excel(config.WAVE2_ENUMERATORS / "correct_assignment.xlsx")

In [3]:
# For labgroupids in the corrections, update the enumerator assignment and treatment status

labgroupids_to_correct = corrections["labgroupid"].tolist()

for labgroupid in labgroupids_to_correct:
    # Get the correct enumerator and treatment status from the corrections dataframe
    for col in ["enum_id", "Treatment Status", "enum_firstname", "enum_lastname", "enum_email"]:
        if col not in corrections.columns:
            raise ValueError(f"Column '{col}' is missing from the corrections dataframe.")
        else:
            correct_value = corrections.loc[corrections["labgroupid"] == labgroupid, col].values[0]
            existing_assignments.loc[existing_assignments["labgroupid"] == labgroupid, col] = correct_value
    
    # Create new col to indicate that not randomly assigned
    existing_assignments.loc[existing_assignments["labgroupid"] == labgroupid, "not_randomly_assigned"] = 1

In [4]:
# Reorder columns for saving assignments file
assignments_order = [
    "labgroupid", "Lab Group", "Faculty", "Institute", 
    "Professor", "Email", "Source", "Treatment Status", 
    "enum_id", "enum_firstname", "enum_lastname", 
    "enum_email", "out_of_sample",
    "not_randomly_assigned"
]

# Save the assignments file
cols_to_save = [col for col in assignments_order if col in existing_assignments.columns]
existing_assignments.to_csv(config.WAVE2_ENUMERATORS / "assignedlabs.csv", index = False, columns = cols_to_save)